# Train two attack-family-specific hybrid models

Instead of one generalist deployed model covering every attack shape,
this trains **two separate** XGBoost + Isolation Forest hybrids (same
25-feature set as the deployed model), each on data matching one
structural attack family:

1. **Volumetric floods** (`models/family_volumetric/`) - single repeated
   packet type at high rate, no completed connection, minimal-to-no
   payload; the goal is raw exhaustion, not interaction. Matches this
   project's live ICMP/SYN/UDP flood simulator scenarios.
2. **Connection / application-layer attacks** (`models/family_connection/`)
   - real connection attempts producing genuine bidirectional traffic:
   full TCP handshake for HTTP-style DoS, per-port connect attempts for
   scanning. Matches this project's live HTTP flood / port scan scenarios.

## Label mapping used (the only place this is decided)

| Family | Source | Labels included | Why |
|---|---|---|---|
| Volumetric floods | CIC-DDoS2019 | `Syn`, `TFTP` | Named as the non-reflection flood types this family is built from. Reflection/amplification types (`DrDoS_*`, plain `UDP`, `UDPLag`, `Portmap`) are **excluded** - their traffic shape differs (spoofed source, real response payloads from a reflector), and only Syn/TFTP were named as the intended fit. |
| Connection/app-layer | CIC-IDS2018 | `DoS attacks-Hulk`, `DoS attacks-GoldenEye`, `DoS attacks-Slowloris`, `DoS attacks-SlowHTTPTest`, `Infilteration` | Hulk/GoldenEye/Slowloris are the named examples; SlowHTTPTest is the same "-style" HTTP-DoS family. `Infilteration` (CIC-IDS2018's own spelling) is the closest available reconnaissance/infiltration analogue - the dataset has no standalone PortScan label. |

Everything else (2018 brute force, web/XSS, SQL injection, Bot; 2019's
`DrDoS_*` reflection types) is **excluded from both** - neither a flood
nor a connection/recon behavior, so including it would just be label
noise. If this mapping should be broader (e.g. `DrDoS_*` should also
count as volumetric), the `VOLUMETRIC_DAY_TAG_SUFFIXES` /
`CONNECTION_2018_LABELS` constants below are the only place to change -
everything downstream is generic.

Each family's benign rows come only from the days/files that contribute
to that family (own-benign-only, extending this project's established
convention from `fetch_ddos2019_sample_25feature.py` - no cross-family
or cross-dataset benign blending).

Architecture/hyperparameters, scaling, and threshold-tuning are
otherwise identical to `retrain_deployed_model.py` (XGBoost n_estimators=300,
max_depth=6, lr=0.08; Isolation Forest n_estimators=500, benign-only fit;
RobustScaler; threshold tuned against `detection/predictor.py`'s actual
`xgb_probability >= threshold OR isolation_prediction` combination logic,
not a blended score) - for continuity and comparability with the deployed
model. Not wired into serving code; this only trains and reports metrics.

**Prerequisites** (same as `retrain_deployed_model.ipynb`):
- `data/cicids2018/` - 28 offset-window files from `fetch_cicids2018_multiday_v2.py`
- `data/cicids2018_full/` - the 3 full days
- `data/ddos2019_sample/combined_sample_25feature.csv` - from `fetch_ddos2019_sample_25feature.py`
- `models/top_features.pkl` - the existing 25-feature set (reused unchanged)


In [ ]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier


## Config - paths, feature set, family definitions

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Same 10-day CIC-IDS2018 source set as retrain_deployed_model.py (28
# spread-out offset windows for the 7 "partial" days, via
# fetch_cicids2018_multiday_v2.py, + 3 full days).
PARTIAL_DIR = PROJECT_ROOT / "data" / "cicids2018"
PARTIAL_DAYS = [
    "02-14-2018_off0.csv", "02-14-2018_off1.csv", "02-14-2018_off2.csv", "02-14-2018_off3.csv",
    "02-15-2018_off0.csv", "02-15-2018_off1.csv", "02-15-2018_off2.csv", "02-15-2018_off3.csv",
    "02-16-2018_off0.csv", "02-16-2018_off1.csv", "02-16-2018_off2.csv", "02-16-2018_off3.csv",
    "02-20-2018_off0.csv", "02-20-2018_off1.csv", "02-20-2018_off2.csv", "02-20-2018_off3.csv",
    "02-21-2018_off0.csv", "02-21-2018_off1.csv", "02-21-2018_off2.csv", "02-21-2018_off3.csv",
    "02-22-2018_off0.csv", "02-22-2018_off1.csv", "02-22-2018_off2.csv", "02-22-2018_off3.csv",
    "02-23-2018_off0.csv", "02-23-2018_off1.csv", "02-23-2018_off2.csv", "02-23-2018_off3.csv",
]
FULL_DIR = PROJECT_ROOT / "data" / "cicids2018_full"
FULL_DAYS = ["02-28-2018.csv", "03-01-2018.csv", "03-02-2018.csv"]

DDOS2019_SAMPLE = PROJECT_ROOT / "data" / "ddos2019_sample" / "combined_sample_25feature.csv"

with open(PROJECT_ROOT / "models" / "top_features.pkl", "rb") as f:
    TOP_FEATURES = pickle.load(f)

DERIVED = {"pkt_rate_ratio", "iat_variation"}
RAW_NEEDED_2018 = [c for c in TOP_FEATURES if c not in DERIVED]

# The two family definitions - see the intro cell for why each list is
# what it is. Change these, and only these, to adjust the split.
VOLUMETRIC_DAY_TAG_SUFFIXES = ("_Syn", "_TFTP")

CONNECTION_2018_LABELS = [
    "DoS attacks-Hulk",
    "DoS attacks-GoldenEye",
    "DoS attacks-Slowloris",
    "DoS attacks-SlowHTTPTest",
    "Infilteration",  # CIC-IDS2018's own spelling
]

TOP_FEATURES


## Loaders (identical to `retrain_deployed_model.py`)

In [ ]:
def load_one_2018_day(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df[df["Label"] != "Label"].reset_index(drop=True)

    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
    df = df.dropna(subset=["Timestamp"])
    df["Binary_Label"] = df["Label"].apply(lambda x: 0 if str(x).strip().lower() == "benign" else 1)

    for col in RAW_NEEDED_2018 + ["Flow Pkts/s", "Fwd Pkts/s", "Flow IAT Mean", "Flow IAT Std"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["pkt_rate_ratio"] = df["Fwd Pkts/s"] / (df["Flow Pkts/s"] + 1)
    df["iat_variation"] = df["Flow IAT Std"] / (df["Flow IAT Mean"] + 1)

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=TOP_FEATURES)
    df["day"] = path.stem
    return df.sort_values("Timestamp").reset_index(drop=True)


def load_ddos2019_sample():
    if not DDOS2019_SAMPLE.exists():
        raise FileNotFoundError(
            f"{DDOS2019_SAMPLE} not found - run fetch_ddos2019_sample_25feature.py first"
        )
    df = pd.read_csv(DDOS2019_SAMPLE, low_memory=False)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    return [g.sort_values("Timestamp").reset_index(drop=True) for _, g in df.groupby("day")]


def per_day_split(day_df):
    attacks = day_df.loc[day_df["Binary_Label"] == 1, "Timestamp"]
    if len(attacks) < 20:
        cutoff = day_df["Timestamp"].quantile(0.8)
    else:
        cutoff = attacks.quantile(0.7)
    train_mask = (day_df["Timestamp"] < cutoff).values
    return train_mask, ~train_mask


## Build the two family-specific datasets

`build_volumetric_dataset()` restricts to the Syn/TFTP day-tag files -
each already own-benign-only per `fetch_ddos2019_sample_25feature.py`,
so no further label filtering is needed once the files are selected.

`build_connection_dataset()` loads all 10 CIC-IDS2018 days, keeps every
day's own benign rows unchanged, but **drops** (not relabels) any attack
row whose `Label` isn't in `CONNECTION_2018_LABELS` - so brute force,
web/XSS, SQL injection, and Bot rows never leak into either class.

In [ ]:
def build_volumetric_dataset():
    all_frames = load_ddos2019_sample()
    frames = [f for f in all_frames if f["day"].iloc[0].endswith(VOLUMETRIC_DAY_TAG_SUFFIXES)]

    if not frames:
        raise ValueError(
            "no Syn/TFTP day tags found in combined_sample_25feature.csv - "
            "check fetch_ddos2019_sample_25feature.py's output log; one of "
            "these two attack-type files may have produced zero usable rows."
        )

    found_tags = sorted(f["day"].iloc[0] for f in frames)
    print(f"  volumetric source files: {found_tags}")
    if not any(t.endswith("_Syn") for t in found_tags):
        print("  WARNING: no Syn file found")
    if not any(t.endswith("_TFTP") for t in found_tags):
        print("  WARNING: no TFTP file found")

    train_parts, test_parts = [], []
    for day_df in frames:
        tr, te = per_day_split(day_df)
        train_parts.append(day_df[tr])
        test_parts.append(day_df[te])
        atk = day_df["Binary_Label"]
        print(f"  {day_df['day'].iloc[0]}: {len(day_df)} rows, "
              f"train={tr.sum()} (attack={atk[tr].sum()}), test={te.sum()} (attack={atk[te].sum()})")

    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


def build_connection_dataset():
    frames = [load_one_2018_day(PARTIAL_DIR / n) for n in PARTIAL_DAYS]
    frames += [load_one_2018_day(FULL_DIR / n) for n in FULL_DAYS]

    train_parts, test_parts = [], []
    total_kept_attacks = 0
    for day_df in frames:
        keep = (day_df["Label"].str.strip() == "Benign") | (day_df["Label"].isin(CONNECTION_2018_LABELS))
        day_df = day_df[keep].reset_index(drop=True)
        if day_df.empty:
            continue

        tr, te = per_day_split(day_df)
        train_parts.append(day_df[tr])
        test_parts.append(day_df[te])
        atk = day_df["Binary_Label"]
        total_kept_attacks += int(atk.sum())
        if atk.sum() > 0:
            print(f"  {day_df['day'].iloc[0]}: {len(day_df)} rows kept, "
                  f"train={tr.sum()} (attack={atk[tr].sum()}), test={te.sum()} (attack={atk[te].sum()})")

    if total_kept_attacks == 0:
        raise ValueError(
            f"no rows matched CONNECTION_2018_LABELS={CONNECTION_2018_LABELS} across any "
            f"loaded day - check the exact Label spellings in your local CSVs."
        )

    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


## Shared train/eval/save

Same methodology as `retrain_deployed_model.py`: RobustScaler + Isolation
Forest fit on benign-only rows, XGBoost on the full set, threshold tuned
by F1-maximizing sweep against the *actual* deployed OR-combination logic
(`xgb_probability >= threshold OR isolation_prediction == 1`), not a
blended score.

In [ ]:
def tune_threshold(y_true, probs):
    return float(max(np.linspace(0, 1, 100), key=lambda v: f1_score(y_true, probs >= v, zero_division=0)))


def eval_hybrid(y_true, xgb_probs, isolation_preds, threshold):
    xgb_pred = (xgb_probs >= threshold).astype(int)
    hybrid_pred = ((xgb_pred == 1) | (isolation_preds == 1)).astype(int)
    return {
        "accuracy": accuracy_score(y_true, hybrid_pred),
        "precision": precision_score(y_true, hybrid_pred, zero_division=0),
        "recall": recall_score(y_true, hybrid_pred, zero_division=0),
        "f1": f1_score(y_true, hybrid_pred, zero_division=0),
    }


def feature_importance_report(names, importances):
    report = sorted(zip(names, [float(i) for i in importances]), key=lambda x: -x[1])
    shortcut_warning = report[0][1] > 0.5
    return report, shortcut_warning


def train_family_model(family_name, dataset_description, train_df, test_df, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 70}\ntraining: {family_name}\n{'=' * 70}")
    print(f"train={len(train_df)} test={len(test_df)}")
    print(f"train attack ratio: {train_df['Binary_Label'].mean():.3f}")
    print(f"test attack ratio:  {test_df['Binary_Label'].mean():.3f}")

    X_train = train_df[TOP_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_train = np.clip(X_train, -1e9, 1e9).astype(np.float64)
    y_train = train_df["Binary_Label"].values

    X_test = test_df[TOP_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = np.clip(X_test, -1e9, 1e9).astype(np.float64)
    y_test = test_df["Binary_Label"].values

    scaler = RobustScaler()
    X_benign_train = X_train[y_train == 0]
    X_benign_scaled = scaler.fit_transform(X_benign_train)
    X_test_scaled = scaler.transform(X_test)

    print("training Isolation Forest (benign-only) ...")
    iso = IsolationForest(
        n_estimators=500, max_samples=512, contamination=0.12,
        max_features=0.8, random_state=42, n_jobs=-1,
    )
    iso.fit(X_benign_scaled)
    isolation_preds_test = (iso.predict(X_test_scaled) == -1).astype(int)

    print("training XGBoost ...")
    xgb = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.08,
        subsample=0.8, colsample_bytree=0.8, gamma=0.1,
        reg_alpha=0.1, reg_lambda=1.0, eval_metric="logloss",
        n_jobs=-1, random_state=42,
    )
    xgb.fit(X_train, y_train)
    xgb_probs_test = xgb.predict_proba(X_test)[:, 1]

    threshold = tune_threshold(y_test, xgb_probs_test)
    metrics = eval_hybrid(y_test, xgb_probs_test, isolation_preds_test, threshold)
    print(f"tuned threshold: {threshold}")
    print(f"hybrid metrics (xgb>=threshold OR isolation) on held-out data: {metrics}")

    imp, shortcut_warning = feature_importance_report(TOP_FEATURES, xgb.feature_importances_)
    print(f"shortcut_warning: {shortcut_warning}")

    isolation_only_recall = recall_score(y_test, isolation_preds_test, zero_division=0)
    xgb_only_pred = (xgb_probs_test >= threshold).astype(int)
    xgb_only_metrics = {
        "accuracy": accuracy_score(y_test, xgb_only_pred),
        "precision": precision_score(y_test, xgb_only_pred, zero_division=0),
        "recall": recall_score(y_test, xgb_only_pred, zero_division=0),
        "f1": f1_score(y_test, xgb_only_pred, zero_division=0),
    }
    print(f"xgb-alone metrics at tuned threshold: {xgb_only_metrics}")
    print(f"isolation-forest-alone recall: {isolation_only_recall}")

    with open(out_dir / "xgb_model.pkl", "wb") as f:
        pickle.dump(xgb, f)
    with open(out_dir / "isolation_forest.pkl", "wb") as f:
        pickle.dump(iso, f)
    with open(out_dir / "scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    with open(out_dir / "threshold.pkl", "wb") as f:
        pickle.dump(threshold, f)
    with open(out_dir / "top_features.pkl", "wb") as f:
        pickle.dump(TOP_FEATURES, f)

    provenance = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "family": family_name,
        "dataset": dataset_description,
        "day_tags": sorted(pd.concat([train_df, test_df])["day"].unique().tolist()),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "hybrid_metrics_held_out": metrics,
        "xgb_alone_metrics_held_out": xgb_only_metrics,
        "isolation_alone_recall_held_out": float(isolation_only_recall),
        "feature_importances": imp,
        "shortcut_warning": shortcut_warning,
        "threshold": threshold,
    }
    with open(out_dir / "provenance.json", "w", encoding="utf-8") as f:
        json.dump(provenance, f, indent=2, default=str)

    print(f"saved artifacts to {out_dir}/")
    return provenance


## Build both datasets

In [ ]:
print("building VOLUMETRIC FLOOD dataset (CIC-DDoS2019 Syn + TFTP) ...")
vol_train, vol_test = build_volumetric_dataset()

print("\nbuilding CONNECTION/APPLICATION-LAYER dataset (CIC-IDS2018 Hulk/GoldenEye/Slowloris/SlowHTTPTest + Infilteration) ...")
conn_train, conn_test = build_connection_dataset()


## Train the volumetric-flood model -> `models/family_volumetric/`

In [ ]:
volumetric_prov = train_family_model(
    "volumetric_floods",
    "CIC-DDoS2019 Syn + TFTP (non-reflection flood types) - own benign only per file",
    vol_train, vol_test,
    PROJECT_ROOT / "models" / "family_volumetric",
)


## Train the connection/application-layer model -> `models/family_connection/`

In [ ]:
connection_prov = train_family_model(
    "connection_application_layer",
    "CIC-IDS2018 DoS-Hulk/GoldenEye/Slowloris/SlowHTTPTest + Infilteration - own benign per day, other attack types excluded",
    conn_train, conn_test,
    PROJECT_ROOT / "models" / "family_connection",
)


## Summary

Not wired into serving code - these artifacts sit at
`models/family_volumetric/` and `models/family_connection/` for review,
same as `models/deployed_v2/` was before promotion.

In [ ]:
for name, prov in [("volumetric_floods", volumetric_prov), ("connection_application_layer", connection_prov)]:
    m = prov["hybrid_metrics_held_out"]
    print(f"{name}: train={prov['train_rows']} test={prov['test_rows']} "
          f"acc={m['accuracy']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} f1={m['f1']:.3f} "
          f"shortcut_warning={prov['shortcut_warning']}")
